# Vietnamese TTS - Google Colab UI
Chạy các cell bên dưới để khởi động giao diện Gradio cho TTS Engine.

Đảm bảo bạn đã upload hoặc clone source code `tts` và đang đứng tại thư mục gốc của project (có chứa `tts_engine.py`, `app.py`...).

In [ ]:
!pip install gradio piper-tts requests

In [ ]:
import gradio as gr
import json
import os
import zipfile
import time
import datetime
import requests
import subprocess
import urllib.parse
import base64

# Hardcoded voices available on GitHub
github_voices = [
    "adam",
    "Bông Cúc",
    "Ngọc Huyền",
    "Yan "
]

def download_model(voice_name):
    try:
        os.makedirs("downloaded_models", exist_ok=True)
        # Handle spaces and special characters in URL
        encoded_name = urllib.parse.quote(voice_name)
        base_url = f"https://raw.githubusercontent.com/kandinz/Omni-TTS/refs/heads/main/onnx/{encoded_name}.onnx"
        
        model_name = f"{voice_name}.onnx"
        model_path = os.path.join("downloaded_models", model_name)
        config_path = model_path + ".json"
        
        # Download onnx
        if not os.path.exists(model_path):
            print(f"Downloading model from {base_url}...")
            r = requests.get(base_url, allow_redirects=True)
            if r.status_code == 200:
                with open(model_path, 'wb') as f:
                    f.write(r.content)
            else:
                return None, f"Lỗi tải model: HTTP {r.status_code}"
                
        # Download config
        config_url = base_url + ".json"
        if not os.path.exists(config_path):
            print(f"Downloading config from {config_url}...")
            r = requests.get(config_url, allow_redirects=True)
            if r.status_code == 200:
                with open(config_path, 'wb') as f:
                    f.write(r.content)
            else:
                return None, f"Lỗi tải config: HTTP {r.status_code}"
                
        return model_path, None
    except Exception as e:
        return None, f"Lỗi khi tải model: {str(e)}"

def process_tts(input_text, voice_name, speed):
    if not input_text.strip():
        return None, None, "", "Vui lòng nhập nội dung văn bản hoặc JSON."

    output_dir = "colab_output"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Download and get model path
    if not voice_name:
        return None, None, "", "Vui lòng chọn giọng đọc."
        
    model_path, err = download_model(voice_name)
    if err:
        return None, None, "", err

    # Check if input is JSON
    is_batch = False
    items = []
    try:
        data = json.loads(input_text)
        if isinstance(data, list) and len(data) > 0 and "text" in data[0]:
            is_batch = True
            items = data
    except json.JSONDecodeError:
        pass
    
    if not is_batch:
        items = [{"filename": f"output_{timestamp}.wav", "text": input_text.strip()}]
        
    generated_files = []
    
    for item in items:
        text = item.get("text", "").strip()
        filename = item.get("filename", f"output_{time.time()}.wav")
        if not text:
            continue
            
        if not filename.endswith(".wav"):
            filename += ".wav"
            
        filepath = os.path.join(output_dir, filename)
        
        try:
            length_scale = 1.0 / speed if speed > 0 else 1.0
            cmd = ["piper", "--model", model_path, "--output_file", filepath, "--length_scale", str(length_scale)]
            
            process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            stdout, stderr = process.communicate(input=text.encode('utf-8'))
            
            if process.returncode == 0 and os.path.exists(filepath):
                generated_files.append(filepath)
            else:
                print(f"Lỗi Piper: {stderr.decode('utf-8', errors='ignore')}")
        except Exception as e:
            print(f"Lỗi khi tạo file {filename}: {e}")
            
    if not generated_files:
        return None, None, "", "Không có file nào được tạo ra."
        
    # Zip all files
    zip_path = os.path.join(output_dir, f"batch_output_{timestamp}.zip")
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for f in generated_files:
            zipf.write(f, os.path.basename(f))
            
    # Generate HTML for audio previews
    audio_html = "<div style='display: flex; flex-direction: column; gap: 15px;'>"
    for f in generated_files:
        try:
            with open(f, "rb") as audio_file:
                b64 = base64.b64encode(audio_file.read()).decode("utf-8")
            filename = os.path.basename(f)
            audio_html += f"<div><p style='margin: 0 0 5px 0; font-weight: bold;'>{filename}</p><audio controls src='data:audio/wav;base64,{b64}' style='width: 100%; height: 40px;'></audio></div>"
        except Exception as e:
            pass
    audio_html += "</div>"
            
    return generated_files, zip_path, audio_html, f"Hoàn tất. Đã tạo thành công {len(generated_files)} file."

def create_ui():
    with gr.Blocks(title="Vietnamese TTS - Colab UI") as app:
        gr.Markdown("# 🎙️ Piper TTS - Google Colab UI")
        gr.Markdown("Công cụ tổng hợp giọng nói sử dụng **Piper TTS**. Hỗ trợ cả chế độ Văn bản thường và JSON (Batch Mode).")
        
        with gr.Row():
            with gr.Column(scale=2):
                input_text = gr.Textbox(
                    label="Nội dung (Text hoặc JSON)",
                    lines=12,
                    placeholder='''Nhập văn bản bình thường hoặc nhập JSON dạng:
[
  {
    "filename": "audio-scene-1.wav",
    "text": "script-scene-1_text"
  }
]'''
                )
                
            with gr.Column(scale=1):
                voice_dropdown = gr.Dropdown(
                    choices=github_voices, 
                    value=github_voices[0] if github_voices else None, 
                    label="Giọng đọc"
                )
                gr.Markdown("*Ghi chú: Lần đầu chọn giọng đọc, hệ thống sẽ tự động tải model từ GitHub về. Các lần sau sẽ dùng bản đã tải (nhanh hơn).*")
                speed = gr.Slider(0.3, 2.0, value=1.0, step=0.1, label="Tốc độ (Speed)")
                
        btn_generate = gr.Button("🚀 Generate TTS", variant="primary")
        
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🎧 Nghe trước")
                preview_html = gr.HTML(label="Nghe trước")
            with gr.Column(scale=1):
                output_files = gr.File(label="Tải từng file (WAV)", file_count="multiple")
                output_zip = gr.File(label="Tải tất cả (ZIP)")
                
        output_msg = gr.Textbox(label="Trạng thái", interactive=False)
            
        btn_generate.click(
            fn=process_tts,
            inputs=[input_text, voice_dropdown, speed],
            outputs=[output_files, output_zip, preview_html, output_msg]
        )
        
    return app

app = create_ui()
app.launch(debug=True, inline=True)
